Подключаем библиотеку pandas

In [1]:
import pandas as pd # подключение библиотеки pandas для работы с датасетом и присваиваю ему псевдоним pd

Считываем наш датасет, выводим все его столбцы количество ненулевых значений, тип данных и размерность датасета
так же считаем долю пропущенных значений

In [2]:
# чтение csv файла, чтобы можно было работать с датасетом
# перебор путей — по требованию задания: ноутбук должен запускаться и в тренажёре (/datasets/),
# и локально, с копией файла рядом с ноутбуком
try:
    df = pd.read_csv('/datasets/new_games.csv')
except FileNotFoundError:
    df = pd.read_csv('new_games.csv')
df.info() # информация о датасете

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16954 non-null  object 
 1   Platform         16956 non-null  object 
 2   Year of Release  16681 non-null  float64
 3   Genre            16954 non-null  object 
 4   NA sales         16956 non-null  float64
 5   EU sales         16956 non-null  object 
 6   JP sales         16956 non-null  object 
 7   Other sales      16956 non-null  float64
 8   Critic Score     8242 non-null   float64
 9   User Score       10152 non-null  object 
 10  Rating           10085 non-null  object 
dtypes: float64(4), object(7)
memory usage: 1.4+ MB


после чтения датасета, можно выделить, есть несколько колонок, которые нуждаются в изменении типа данных, так как в них хранятся числа, но по какой-то причине они строкового типа данных, также хочу отметить, что в основном пропуски присутствуют в рейтинге игр, это может быть связано с тем, что старые игры не оценивались и их количество, скорее всего, преобладает, либо некоторые игры были непопулярны и не получили оценку

я оставляю путь к файлу datasets/new_games.csv, потому что у меня ругается на filenotfounderror /datasets/new_games.csv не работает, без первого слэша всё ок

---

In [3]:
# приведение year of release к Int16, потому что int16 numpy не может хранить NaN, а Int16 pandas может, так же приведение года к целочисленному типу данных обусловлено более понятным для работы
# нежели число с плавающей точкой
df['Year of Release'] = pd.to_numeric(df['Year of Release'], errors='coerce').astype('Int16')

In [4]:
df['Year of Release']

0        2006
1        1985
2        2008
3        2009
4        1996
         ... 
16951    2016
16952    2006
16953    2016
16954    2003
16955    2016
Name: Year of Release, Length: 16956, dtype: Int16

Перевёл год к целочисленному типу данных, потому что нам нужен только год, а не месяц и день, вдруг дата не совпадёт с реальной датой выхода игры, поэтому к дате решил не переводить, так же вышла проблема, из-за того что np.nan поддерживает только float32,64, точнее только он может хранить и числа и np.nan, в отличии от np.int8,16,32,64, поэтому выбрал Int8,16,32,64 — это тип данных pandas, он может хранить и числа, и NaN, в отличие от numpy-типов; а конструкция downcast не срабатывала, потому что приводила к numpy dtype

Приступаю к переводу "NA sales", "EU sales", "JP sales" и "Other sales" к float32, после чего переведу все колонки в snake case

In [5]:
# использую генератор, чтобы пройтись по всем названиям и перевести их к snake case
df.columns = [s.lower().replace(' ', '_') for s in df.columns]

In [6]:
df.info() # проверка приведения полей к snake_case

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16954 non-null  object 
 1   platform         16956 non-null  object 
 2   year_of_release  16681 non-null  Int16  
 3   genre            16954 non-null  object 
 4   na_sales         16956 non-null  float64
 5   eu_sales         16956 non-null  object 
 6   jp_sales         16956 non-null  object 
 7   other_sales      16956 non-null  float64
 8   critic_score     8242 non-null   float64
 9   user_score       10152 non-null  object 
 10  rating           10085 non-null  object 
dtypes: Int16(1), float64(3), object(7)
memory usage: 1.3+ MB


In [27]:
# приведение other_sales к float для уменьшения веса датасета, так как будет присвоен float32
df['na_sales'] = pd.to_numeric(df['na_sales'], errors='coerce', downcast='float')
# приведение other_sales к float
df['eu_sales'] = pd.to_numeric(df['eu_sales'], errors='coerce', downcast='float')
# приведение other_sales к float
df['jp_sales'] = pd.to_numeric(df['jp_sales'], errors='coerce', downcast='float')
# приведение other_sales к float для уменьшения веса датасета, так как будет присвоен float32
df['other_sales'] = pd.to_numeric(df['other_sales'], errors='coerce', downcast='float')

In [8]:
# приведение critic_score к float
df['critic_score'] = pd.to_numeric(df['critic_score'], downcast='float')
# приведение user_score к float
# errors='coerce' введено потому, что внутри были строчные значения и их заменил на NaN
df['user_score'] = pd.to_numeric(df['user_score'], errors='coerce', downcast='float')

In [9]:
df.info() # проверка присвоения типов данных

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16954 non-null  object 
 1   platform         16956 non-null  object 
 2   year_of_release  16681 non-null  Int16  
 3   genre            16954 non-null  object 
 4   na_sales         16956 non-null  float32
 5   eu_sales         16950 non-null  float32
 6   jp_sales         16952 non-null  float32
 7   other_sales      16956 non-null  float32
 8   critic_score     8242 non-null   float32
 9   user_score       7688 non-null   float32
 10  rating           10085 non-null  object 
dtypes: Int16(1), float32(6), object(4)
memory usage: 977.1+ KB


после перевода всех колонок с ошибочным типом данных, обнаружил, что количество пропусков в user_score увеличилось практически вдвое, это значит, что были текстовые записки, которые не были числами, было 'tbd', что означает "подлежит уточнению"

In [10]:
df.isna().sum() / df.notna().sum() # относительное значение

name               0.000118
platform           0.000000
year_of_release    0.016486
genre              0.000118
na_sales           0.000000
eu_sales           0.000354
jp_sales           0.000236
other_sales        0.000000
critic_score       1.057268
user_score         1.205515
rating             0.681309
dtype: float64

In [11]:
df.isna().mean() #среднее количество пропусков в каждом столбце

name               0.000118
platform           0.000000
year_of_release    0.016218
genre              0.000118
na_sales           0.000000
eu_sales           0.000354
jp_sales           0.000236
other_sales        0.000000
critic_score       0.513918
user_score         0.546591
rating             0.405225
dtype: float64

разобрался, чем отличается df.isna().sum() / df.notna().sum() от df.isna().mean(): первый способ даёт соотношение между пропусками и не пропусками, второй — долю пропусков от общего числа строк, что для оценки качества данных корректнее. Первый вариант применяется реже

In [12]:
df.isna().sum() # абсолютное значение

name                  2
platform              0
year_of_release     275
genre                 2
na_sales              0
eu_sales              6
jp_sales              4
other_sales           0
critic_score       8714
user_score         9268
rating             6871
dtype: int64

После анализа пропусков можно сделать несколько выводов: для старых игр либо не было оценок, либо датасет при передаче был повреждён, так же могу отметить, что есть пропуски в колонке year_of_release, но количество пропусков небольшое, всего 1.6%, но заменить на какое-то среднее или на индикатор будет не целесообразно, так как при передаче датасета это может запутать, что значит среднее или индикатор в столбце, было принято решение оставить NaN (это я пообщался с преподавателем на паре в универе и он сказал, чтобы избежать недопонимания, лучше ничего в столбце с годом не менять и оставить так, потому что человек сразу поймёт, что данных о годе не было), также есть пропуски в названии игры и жанре, и эти пропуски находятся в одной строке, так же не заменял средним или индикатором в eu_sales и jp_sales, так как дляы реальных результатов среднее может быть либо огромным, либо слишком маленьким, а индикатор может запутать, например, если он будет -1

In [13]:
# проверка двух пропусков в строках genre и name, чтобы узнать, несут они пользу в анализе или нет
df[df['genre'].isna()] 

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
661,NaN,GEN,1993,NaN,1.78,0.53,0.00,0.08,NaN,NaN,NaN
14439,NaN,GEN,1993,NaN,0.00,0.00,0.03,0.00,NaN,NaN,NaN


Пропуски в столбцах с оценками не заполняются и не удаляются. Большинство из них структурные — игра не получила оценку критиков или пользователей, - либо игра просто старая и тогда никто их не оценивал

In [14]:
df['name'].unique() # проверка поля name на уникальные значения

array(['Wii Sports', 'Super Mario Bros.', 'Mario Kart Wii', ...,
       'Woody Woodpecker in Crazy Castle 5', 'LMA Manager 2007',
       'Haitaka no Psychedelica'], dtype=object)

In [15]:
df['platform'].unique() # проверка поля platform на уникальные значения

array(['Wii', 'NES', 'GB', 'DS', 'X360', 'PS3', 'PS2', 'SNES', 'GBA',
       'PS4', '3DS', 'N64', 'PS', 'XB', 'PC', '2600', 'PSP', 'XOne',
       'WiiU', 'GC', 'GEN', 'DC', 'PSV', 'SAT', 'SCD', 'WS', 'NG', 'TG16',
       '3DO', 'GG', 'PCFX'], dtype=object)

In [16]:
df['genre'].unique() # проверка поля genre на уникальные значения

array(['Sports', 'Platform', 'Racing', 'Role-Playing', 'Puzzle', 'Misc',
       'Shooter', 'Simulation', 'Action', 'Fighting', 'Adventure',
       'Strategy', nan, 'MISC', 'ROLE-PLAYING', 'RACING', 'ACTION',
       'SHOOTER', 'FIGHTING', 'SPORTS', 'PLATFORM', 'ADVENTURE',
       'SIMULATION', 'PUZZLE', 'STRATEGY'], dtype=object)

In [17]:
df['rating'].unique() # проверка поля rating на уникальные значения

array(['E', nan, 'M', 'T', 'E10+', 'K-A', 'AO', 'EC', 'RP'], dtype=object)

после проверки столбцов со строковым типом данных, был сделан вывод, что неявные дубликаты могут содержаться в названии игры и содержаться точно в жанре игры, поэтому только они будут приведены к нижнему регистру, а приведение к верхнему регистру колонку "rating" не имеет смысла, так как в ней не будет неявных дубликатов и она и так верхнего региста

In [18]:
df['name'] = df['name'].str.lower() # приведение поля name к нижнему регистру
df['genre'] = df['genre'].str.lower() # приведение поля genre к нижнему регистру

In [19]:
df[df.duplicated()] # просмотр всех дубликатов и их количество

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
268,batman: arkham asylum,PS3,2009,action,2.24,1.31,0.07,0.61,91.0,8.9,T
368,james bond 007: agent under fire,PS2,2001,shooter,1.90,1.13,0.10,0.41,72.0,7.9,T
717,god of war: ascension,PS3,2013,action,1.23,0.63,0.04,0.35,80.0,7.5,M
823,wipeout: the game,Wii,2009,misc,1.94,0.00,0.00,0.12,NaN,NaN,NaN
848,rayman raving rabbids: tv party,Wii,2008,misc,0.72,1.08,0.00,0.23,73.0,7.7,E10+
...,...,...,...,...,...,...,...,...,...,...,...
16671,fullmetal alchemist: prince of the dawn,Wii,2009,adventure,0.00,0.00,0.01,0.00,NaN,NaN,NaN
16753,routes pe,PS2,2007,adventure,0.00,0.00,0.01,0.00,NaN,NaN,NaN
16799,transformers: prime,Wii,2012,action,0.00,0.01,0.00,0.00,NaN,NaN,NaN
16912,metal gear solid v: the definitive experience,XOne,2016,action,0.01,0.00,0.00,0.00,NaN,NaN,M


In [20]:
df.drop_duplicates(keep='first', inplace=True) # очистка датасета от дупликатов

после проверки дубликатов, выяснилось, что их было не так много, а именно 241 строка, после чего удалил их и оставил только первое вхождение

In [21]:
df.info() # проверка работы метода drop_duplicates

<class 'pandas.core.frame.DataFrame'>
Int64Index: 16715 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16713 non-null  object 
 1   platform         16715 non-null  object 
 2   year_of_release  16446 non-null  Int16  
 3   genre            16713 non-null  object 
 4   na_sales         16715 non-null  float32
 5   eu_sales         16709 non-null  float32
 6   jp_sales         16711 non-null  float32
 7   other_sales      16715 non-null  float32
 8   critic_score     8137 non-null   float32
 9   user_score       7590 non-null   float32
 10  rating           9949 non-null   object 
dtypes: Int16(1), float32(6), object(4)
memory usage: 1.1+ MB


после подготовки датасета было удалено в абсолютном значении 241 строка, в относительном - 0.014213 или 1.4% всего датасета, после этого можно сделать вывод: дубликаты были удалены, потому что они искажают количества игр и их продажи

In [22]:
def review_critic(row):
    if pd.isna(row['critic_score']): # проверка на присутствие рейтинга
        return 'рейтинг отсутствует'
    elif 80 <= row['critic_score'] <= 100:
        return 'высокая оценка'
    elif 30 <= row['critic_score'] < 80:
        return 'средняя оценка'
    return 'низкая оценка'

def review_user(row):
    if pd.isna(row['user_score']): # проверка на присутствие рейтинга
        return 'рейтинг отсутствует'
    elif 8 <= row['user_score'] <= 10:
        return 'высокая оценка'
    elif 3 <= row['user_score'] < 8:
        return 'средняя оценка'
    return 'низкая оценка'


df_actual = (df[(df['year_of_release'] >= 2000) & (df['year_of_release'] <= 2013)]).reset_index(drop=True)
df_actual['critic_rating'] = df_actual.apply(review_critic, axis=1)
df_actual['user_rating'] = df_actual.apply(review_user, axis=1)

In [23]:
df_actual # проверка работы по создания колонок с оценкой критиков и игроков по критериям оценивания

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating,critic_rating,user_rating
0,wii sports,Wii,2006,sports,41.360001,28.959999,3.77,8.45,76.0,8.0,E,средняя оценка,высокая оценка
1,mario kart wii,Wii,2008,racing,15.680000,12.760000,3.79,3.29,82.0,8.3,E,высокая оценка,высокая оценка
2,wii sports resort,Wii,2009,sports,15.610000,10.930000,3.28,2.95,80.0,8.0,E,высокая оценка,высокая оценка
3,new super mario bros.,DS,2006,platform,11.280000,9.140000,6.50,2.88,89.0,8.5,E,высокая оценка,высокая оценка
4,wii play,Wii,2006,misc,13.960000,9.180000,2.93,2.84,58.0,6.6,E,средняя оценка,средняя оценка
...,...,...,...,...,...,...,...,...,...,...,...,...,...
12776,men in black ii: alien escape,GC,2003,shooter,0.010000,0.000000,0.00,0.00,NaN,NaN,T,рейтинг отсутствует,рейтинг отсутствует
12777,woody woodpecker in crazy castle 5,GBA,2002,platform,0.010000,0.000000,0.00,0.00,NaN,NaN,NaN,рейтинг отсутствует,рейтинг отсутствует
12778,score international baja 1000: the official game,PS2,2008,racing,0.000000,0.000000,0.00,0.00,NaN,NaN,NaN,рейтинг отсутствует,рейтинг отсутствует
12779,lma manager 2007,X360,2006,sports,0.000000,0.010000,0.00,0.00,NaN,NaN,NaN,рейтинг отсутствует,рейтинг отсутствует


In [24]:
# группирую датасет по полю platform, чтобы после пройтись по именам игр,
# посчитать их количество, отсортировать эти значения и вывести топ-7 компании по выпущенным играм
df_actual.groupby('platform')['name'].count().sort_values(ascending=False).head(7)

platform
PS2     2127
DS      2120
Wii     1275
PSP     1180
X360    1121
PS3     1087
GBA      811
Name: name, dtype: int64

In [25]:
df = df.reset_index(drop=True)
df

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
0,wii sports,Wii,2006,sports,41.360001,28.959999,3.77,8.45,76.0,8.0,E
1,super mario bros.,NES,1985,platform,29.080000,3.580000,6.81,0.77,NaN,NaN,NaN
2,mario kart wii,Wii,2008,racing,15.680000,12.760000,3.79,3.29,82.0,8.3,E
3,wii sports resort,Wii,2009,sports,15.610000,10.930000,3.28,2.95,80.0,8.0,E
4,pokemon red/pokemon blue,GB,1996,role-playing,11.270000,8.890000,10.22,1.00,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
16710,samurai warriors: sanada maru,PS3,2016,action,0.000000,0.000000,0.01,0.00,NaN,NaN,NaN
16711,lma manager 2007,X360,2006,sports,0.000000,0.010000,0.00,0.00,NaN,NaN,NaN
16712,haitaka no psychedelica,PSV,2016,adventure,0.000000,0.000000,0.01,0.00,NaN,NaN,NaN
16713,spirits & spells,GBA,2003,platform,0.010000,0.000000,0.00,0.00,NaN,NaN,NaN


# Вывод

По работе можно сделать вывод: была проделана предобработка датасета перед будущим его анализом, были найдены дубликаты, явные и неявные, и удалены из датасета, также были повторены знания, полученные за спринт, узнал много нового, также потихоньку начинаю читать документацию по различным методам, например, errors в to_numeric, который помог привести некоторые данные в нужный мне тип данных, при этом избежав проблем, потому что были в численных столбцах строки, которые благодаря errors='coerce' перевелись в NaN, также узнал, что благодаря downcast в to_numeric pandas переводит в тип данных numpy, а не свой, благодаря чему возникала проблема с переводом из строкового типа данных в float столбца user_score — на этом и спотыкался перевод столбца user_score из строкового типа в float, пока не дочитал документацию; по той же причине перевёл столбец year_of_release в целочисленный тип данных, потому что не мог благодаря обычному методу to_numeric его перевести, так как numpy int не может хранить в себе NaN, а пандас Int может, поэтому одна строка типа данных pandas